[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/refactor/target-layout/notebooks/lin-elastic_strain.ipynb)

# Two-Phase Composite RVE — Linear-Elastic Strain Solve

A minimal walkthrough of FFTjax's strain-based Newton-CG elastic solver
(`problems.mechanics.solve_mechanics`) on a **two-phase composite**: a glass-fiber
reinforcement in an epoxy matrix, arranged in a square-packed pattern via
`generation.rve.make_square_composite_rve`, under a prescribed macroscopic strain.

We want to solve the mechanical equilibrium problem on this composite subject to its governing
PDE constraints:

$$
\nabla \cdot \sigma(\mathbf{x}) = 0, \qquad
\sigma = \mathbb{C}(\mathbf{x}):\varepsilon, \qquad
\varepsilon = \tfrac{1}{2}\big(\nabla u + \nabla u^\top\big)
$$

on a periodic domain, with the macroscopic average strain $\langle\varepsilon(\mathbf{x})\rangle_\Omega = \bar\varepsilon$
prescribed below.

The solver is the Krylov-based (CG-accelerated) Lippmann-Schwinger scheme: the periodic
Lippmann-Schwinger equation, discretized via trigonometric collocation and solved directly by
conjugate gradients against a fixed reference-medium Green's operator. See
[`notes/LEGACY_WORKFLOW.md`](../notes/LEGACY_WORKFLOW.md) for the full derivation.

Because the two phases have a large stiffness contrast (~23x), the reference-medium correction is
nontrivial — the Newton-CG solve actually iterates, redistributing stress between the stiff fibers
and the compliant matrix. For the same reason we use Willot's `rotated` frequency scheme for the
Green's operator rather than the `standard` one: besides avoiding the 45° anisotropy bias of the
naive DFT discretization, it converges markedly better under high stiffness contrast.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git@refactor/target-layout
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import os
import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

`generation.rve.make_square_composite_rve` builds a square-packed 2-fiber RVE: a matrix phase with
circular fiber cross-sections arranged on a square lattice, extruded along Z into a 3-D voxel grid. Here we use a 10-voxel-thick RVE, with a fiber volume fraction of 0.5, fiber radius of 5 μm, and voxel spacing of 0.2 μm.

Setting `nz=1` reduces the geometry to a 2-D-like slab (uniformly extruded along Z); combined with a macroscopic strain that has no out-of-plane (Z) components -- as prescribed below -- this gives a plane-strain solve.

In [ ]:
from generation.rve import make_square_composite_rve

phi     = 0.5      # target fiber volume fraction
r_fiber = 0.005     # fiber radius [mm]
dx      = 0.0002    # target voxel size [mm]

phase_np, n, L, phi_act = make_square_composite_rve(
    phi=phi,
    r_fiber=r_fiber,
    dx=dx,
    N_min=32,       # minimum number of voxels in x, y direction 
    nz=1,           # number of voxels in z direction (thickness) / along fiber axis
)


print("grid n :", n)
print("total voxels Nv:", int(np.prod(n)))
print("domain L [mm]:", tuple(f"{float(Li):.5g}" for Li in L))
print("fiber volume fraction (actual):", f"{phi_act:.4f}")

In [ ]:
# Vizualize the fibre cross-section in the XY plane (Z=0)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r")
ax.set_title(f"Fiber cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [voxel]")
ax.set_ylabel("y [voxel]")
plt.show()

## Materials

A glass fiber in an epoxy matrix — a common, high-contrast (~23x stiffness ratio) composite.

`phase_np` comes back from `make_square_composite_rve` shaped like the physical voxel grid,
`(nx, ny, nz)`. Every per-voxel field inside the solver — `C_field`, strain, stress, `phase`
itself — is instead stored flattened along one trailing axis of length `Nv = nx*ny*nz`, not as
`(nx, ny, nz)`. That's what lets the same tensor-contraction code (`einsum("ijklm,klm->ijm", ...)`
for $\mathbb{C}:\varepsilon$) and the phase-based gather in `assemble_C_field`
(`C_stack[..., phase]`) work unchanged regardless of the grid's actual shape — 2-D or 3-D, any
`nz` — instead of a separate code path per dimensionality. The `(nx, ny, nz)` grid shape only
reappears where it's physically needed: inside the FFT step (reshaped back internally,
transparently to us here) and for plotting/export (`field_to_grid`, used in post-processing
below).

In [ ]:
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from materialmodels.assembly import describe_materials

matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")
materials = [matrix, fiber]   # index 0 = matrix, 1 = fibre -- matches the phase labels below

phase = jnp.array(phase_np.reshape(-1))   # (nx,ny,nz) -> (Nv,), see markdown above

describe_materials(materials)

`solve_mechanics(n, L, phase, materials, eps_bar, ...)` is the one-call entry point: it assembles
$\mathbb{C}(\mathbf{x})$ from `materials`/`phase`, picks the reference medium (the phase-average
Lamé parameters $\lambda_0, \mu_0$ — a reasonable choice when neither phase dominates), builds the
Green's operator, and runs the Newton-CG solve. Its `stepping` argument picks a single full-load
solve (the default, `stepping="single"`) or a load-stepped one; either way it returns
`list[IncrementResult]` — one element here, at `t=1.0` — so `results[0].solution` is the
`ElasticitySolution`, whose `.eps`/`.sigma`/`.delta`/`.converged` attributes we read below (it
also carries an `.eps_bar` field, `None` here — only `DisplacementBasedSolver`'s mixed-BC solve
populates it).

Under the hood (full derivation in [`notes/LEGACY_WORKFLOW.md`](../notes/LEGACY_WORKFLOW.md)), it
solves the periodic Lippmann-Schwinger equation for the total strain field,

$$
\varepsilon(x) + \Gamma_0 * \big[(\mathbb{C}(x)-\mathbb{C}_0):\varepsilon(x)\big] = \bar\varepsilon,
$$

by splitting $\varepsilon = \varepsilon_0 + \Delta\varepsilon$ with $\varepsilon_0 = \bar\varepsilon$
uniform. Since $\Gamma_0 * (\mathbb{C}_0 : \Delta\varepsilon) = \Delta\varepsilon$ for any zero-mean
field and $\hat\Gamma_0(0) = 0$, the $\mathbb{C}_0$-dependence cancels exactly, leaving a linear
system for the correction alone:

$$
\underbrace{\Gamma_0 * (\mathbb{C}:\Delta\varepsilon)}_{A(\Delta\varepsilon)}
\;=\;
\underbrace{-\,\Gamma_0 * (\mathbb{C}:\varepsilon_0)}_{b},
$$

solved by CG to give $\Delta\varepsilon$, then $\varepsilon = \varepsilon_0 + \Delta\varepsilon$
and $\sigma = \mathbb{C}:\varepsilon$. `scheme="rotated"` uses Willot's effective frequencies for
the Green's operator instead of the raw DFT ones — removes the 45° anisotropy bias and converges
markedly better at this composite's stiffness contrast.

Only `n`/`L` go in -- `solve_mechanics` derives `dx` internally wherever it needs it (same
`(n, L)` convention as `GreenOperatorBasic`/`Willot`). Post-processing below derives its own `dx`
from `(n, L)` too, for the one thing that still wants it directly (plot extent);
`compute_displacement` and `IncrementalWriter` both now take `(n, L)` as well, not `dx`, so
there's no `xi_flat` to build and only one `dx` left to carry around, purely for plotting.

In [ ]:
from problems.mechanics import solve_mechanics

# we apply a small shear strain in the XY plane, with zero normal strains
eps_bar = jnp.array([
    [0.0, 1.0e-3, 0.0],
    [1.0e-3, 0.0, 0.0],
    [0.0,    0.0, 0.0],
])

results = solve_mechanics(
    n, L, phase, materials, eps_bar,
    scheme="rotated", toler_lin=1e-6, maxiter=1000,
)
sol = results[0].solution
eps, sigma, delta, converged = sol.eps, sol.sigma, sol.delta, sol.converged

print("converged    :", bool(converged))
print("tau_xy (avg) :", f"{float(jnp.mean(sigma[1, 0])):.3f}", "MPa")
print("G_xy (avg)   :", f"{float(jnp.mean(sigma[1, 0]))/(2*float(jnp.mean(eps[1, 0]))):.3f}", "MPa")

The Newton-CG solve takes real iterations to converge — the correction field is doing real work redistributing stress between the stiff fibres and the compliant matrix.

## Post-processing

Now we can visualize the results and also export them as a `.xdmf`/`.h5` pair for further
post-processing in ParaView or other visualization software, via FFTjax's `IncrementalWriter`
(the project-wide standard for field-data output). Every field here -- displacement, strain,
stress, phase -- is evaluated on the same voxel grid, so all of them are written voxel-centered
(`Center="Cell"`); there's no FEM-style node/cell split in this spectral scheme, so there's nothing
to gain from writing displacement at a different resolution than everything else.

In [ ]:
from post.fields import field_to_grid, von_mises, compute_displacement, to_voigt

# Post-processing
# return the fields to a 3-D grid for visualization and export
eps_grid   = field_to_grid(eps, n)
sigma_grid = field_to_grid(sigma, n)
u_grid     = compute_displacement(eps, eps_bar, n, L)
vm_grid    = von_mises(sigma_grid)

eps_voigt   = to_voigt(eps_grid).astype(np.float64)
sigma_voigt = to_voigt(sigma_grid).astype(np.float64)

display the stress and strain field in matplotlib.

In [ ]:
VOIGT_LABELS = ["x", "y", "z", "xy", "xz", "yz"]
extent = [0.0, L[0] , 0.0, L[1]]  # physical [mm] extent, binned by voxel size dx
CBAR_KW = dict(fraction=0.046, pad=0.04)  # match colorbar height to a square imshow panel --
                                          # without this, fig.colorbar sizes to the axes'
                                          # unshrunk bounding box, not the square image imshow
                                          # actually renders inside it, so it comes out much taller
    
fig, axes = plt.subplots(3, 3, figsize=(10, 9))

im = axes[0, 0].imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r", extent=extent)
axes[0, 0].set_title("Fiber phase")
fig.colorbar(im, ax=axes[0, 0], **CBAR_KW)

im = axes[0, 1].imshow(u_grid[:, :, 0, 0].T, origin="lower", cmap="plasma", extent=extent)
axes[0, 1].set_title(r"Displacement $u_x$ [mm]")
fig.colorbar(im, ax=axes[0, 1], format="%.1e", **CBAR_KW)

im = axes[0, 2].imshow(u_grid[:, :, 0, 1].T, origin="lower", cmap="plasma", extent=extent)
axes[0, 2].set_title(r"Displacement $u_y$ [mm]")
fig.colorbar(im, ax=axes[0, 2], format="%.1e", **CBAR_KW)

for idx, i in enumerate([0, 1, 3]):
    eps_plot = eps_voigt[:, :, 0, i]
    label = rf"$\varepsilon_{{{VOIGT_LABELS[i]}}}$"
    if i == 3:  # shear component: report engineering shear strain gamma = 2*epsilon
        label = rf"$\gamma_{{{VOIGT_LABELS[i]}}}$"

    im = axes[1, idx].imshow(eps_plot.T, origin="lower", cmap="plasma", extent=extent)
    axes[1, idx].set_title(f"Strain {label}")
    fig.colorbar(im, ax=axes[1, idx], format="%.1e", **CBAR_KW)

    label = rf"$\sigma_{{{VOIGT_LABELS[i]}}}$"
    if i == 3:
        label = rf"$\tau_{{{VOIGT_LABELS[i]}}}$"
    im = axes[2, idx].imshow(sigma_voigt[:, :, 0, i].T, origin="lower", cmap="plasma", extent=extent)
    axes[2, idx].set_title(f"Stress {label}")
    fig.colorbar(im, ax=axes[2, idx], format="%.1f", **CBAR_KW)

for ax in axes.flat:
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")

plt.tight_layout()
plt.show()

## Export the fields to XDMF/HDF5



In [ ]:
from utils.io.xdmf_writer import IncrementalWriter

output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)

with IncrementalWriter(f"{output_dir}/composite_rve_", grid_shape=n, grid_length=L) as w:
    w.write_increment(1, {
        "phase":        phase_np.astype(np.float64),
        "displacement": u_grid.astype(np.float64),
        "strain":       eps_voigt.astype(np.float64),
        "stress":       sigma_voigt.astype(np.float64),
        "von_mises":    vm_grid.astype(np.float64),
    }, time=1.0)

print(f"Wrote {output_dir}/composite_rve_.h5")
print(f"      {output_dir}/composite_rve_.xdmf")
print("Open the .xdmf in ParaView with the 'Xdmf3ReaderT' reader.")

## Comparison: smoothing the phase interface (weak vs. sharp boundary)

`phase` above is a hard partition — every voxel is assigned *entirely* matrix or *entirely*
fiber, even though the true circular fiber boundary cuts through many voxels only partially. That
staircased, discontinuous jump in $\mathbb{C}(\mathbf{x})$ at the interface is a numerical
artifact of the voxelization, not the real geometry.

`preprocessing.averaging.ArithmeticAveraging`
([`notes/TARGET_LAYOUT.md`](../notes/TARGET_LAYOUT.md)'s planned `VoxelAveraging`)
fixes this: instead of the hard 0/1 `phase` field, we compute the *true sub-voxel fiber-area
fraction* per voxel (via supersampling) and blend $\mathbb{C}_\text{fiber}$/$\mathbb{C}_\text{matrix}$
by that fraction — a smooth, "weak" transition band exactly one voxel wide around the interface,
instead of a sharp jump.

In [ ]:
from preprocessing.averaging import ArithmeticAveraging
from operators.green import GreenOperatorWillot
from solvers.elliptic.vector.lippmann_schwinger import LippmannSchwingerSolver

# --- per-voxel fiber volume fraction via supersampling (sub-voxel area fraction) ---
factor  = 8                       # supersampling factor per axis
N       = n[0]
N_fine  = factor * N
L_side  = float(L[0])

xs_fine = (np.arange(N_fine) + 0.5) / N_fine * L_side
Xf, Yf  = np.meshgrid(xs_fine, xs_fine, indexing="ij")

def _circle_fine(cx, cy):
    dxg = Xf - cx; dxg -= L_side * np.round(dxg / L_side)
    dyg = Yf - cy; dyg -= L_side * np.round(dyg / L_side)
    return dxg**2 + dyg**2 < r_fiber**2

fiber_fine  = _circle_fine(0.5 * L_side, 0.5 * L_side) | _circle_fine(0.0, 0.0)   # (N_fine, N_fine)
vf_fiber_2d = fiber_fine.astype(np.float64).reshape(N, factor, N, factor).mean(axis=(1, 3))  # (N, N)
vf_fiber    = jnp.asarray(np.repeat(vf_fiber_2d[:, :, None], n[2], axis=2).reshape(-1))       # (Nv,)

n_interface = int(jnp.sum((vf_fiber > 0) & (vf_fiber < 1)))
print(f"interface voxels (0 < vf < 1): {n_interface} / {phase.shape[0]}")

# --- smoothed C_field: blend C_fiber/C_matrix by the sub-voxel fraction ---
C_field_smooth = ArithmeticAveraging().average(fiber.stiffness_tensor(), matrix.stiffness_tensor(), vf_fiber)

# --- solve with the smoothed field, same reference medium/Green's operator solve_mechanics uses ---
lam0     = sum(m.lam for m in materials) / len(materials)
mu0      = sum(m.mu for m in materials) / len(materials)
dx_grid  = tuple(Li / ni for Li, ni in zip(L, n))
green_op = GreenOperatorWillot(n, L, lam0, mu0, dx_grid)
solver   = LippmannSchwingerSolver(n, green_op, toler_lin=1e-6, maxiter=1000)

sol_smooth = solver.solve(C_field_smooth, eps_bar)
sigma_smooth, converged_smooth = sol_smooth.sigma, sol_smooth.converged

tau_sharp  = float(jnp.mean(sigma[1, 0]))
tau_smooth = float(jnp.mean(sigma_smooth[1, 0]))

print("converged (smoothed interface):", bool(converged_smooth))
print("tau_xy sharp    interface:", f"{tau_sharp:.3f}", "MPa")
print("tau_xy smoothed interface:", f"{tau_smooth:.3f}", "MPa")
print("relative difference     :", f"{abs(tau_smooth - tau_sharp) / tau_sharp * 100:.3f} %")

## Next steps

- Sweep grid size for the composite RVE above — the [Benchmark](https://choROPeNt.github.io/FFTjax/documentation/benchmark#linear-elastic-strain-solve) page does exactly this and times it.
- [`lin-elastic_mixed-BC.ipynb`](./lin-elastic_mixed-BC.ipynb) covers the same geometry/materials
  under a free-lateral-surface uniaxial-*stress* condition instead — a more realistic mechanical
  test. It uses the displacement-based solver (`solvers.elliptic.vector.displacement_based`),
  now available on the new `materialmodels`/`solvers.elliptic` layout via
  `solve_mechanics(..., formulation="displacement", control=...)`.